# Kaggle: SegFormer-B2 — TSR vs PCA3 vs PPT

Три отдельные ячейки обучают модели на Kaggle `manual_mask`. Kaggle validation
выбирает checkpoint и threshold; Yandex/TPU используется только как внешний
финальный test с одной общей `base_mask.png`.

Перед `Run All` добавьте **только** `ziangwei/irt-pvc-depth` через **Add Input**,
включите GPU и Internet. Yandex MAT автоматически скачиваются по публичной
ссылке в `/kaggle/temp`, а каноническая `base_mask.png` — из общего GitHub-репозитория.
`kaggle.json` и отдельный Yandex Input не нужны.


In [ ]:
# Установка зависимостей и распаковка встроенного кода проекта.
import base64, io, os, subprocess, sys, tarfile
from pathlib import Path

PINS = [
    "albumentations==2.0.8", "opencv-python-headless==4.12.0.88",
    "scikit-learn==1.9.0", "PyYAML==6.0.3",
    "matplotlib==3.11.1", "transformers==5.15.0",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *PINS])

PROJECT_DIR = Path("/kaggle/working/thermal-segformer")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
PAYLOAD = """..."""
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(PAYLOAD)), mode="r:gz") as archive:
    root = PROJECT_DIR.resolve()
    for member in archive.getmembers():
        destination = (PROJECT_DIR / member.name).resolve()
        assert destination == root or root in destination.parents, member.name
    archive.extractall(PROJECT_DIR)

os.chdir(PROJECT_DIR)
sys.path.insert(0, str(PROJECT_DIR))
assert Path("train_segformer.py").exists()
assert Path("configs/augmentation_tsr.yaml").exists()
print("Embedded training bundle: ed2d661260db")
print("Project directory:", PROJECT_DIR)


In [ ]:
import torch, albumentations, cv2, transformers
print('torch', torch.__version__, '| albumentations', albumentations.__version__)
print('opencv', cv2.__version__, '| transformers', transformers.__version__)
assert torch.cuda.is_available(), 'Включите GPU в Kaggle Settings'
print('GPU:', torch.cuda.get_device_name(0))
print('CUDA devices:', torch.cuda.device_count())
torch.backends.cudnn.benchmark = True
torch.set_float32_matmul_precision('high')

In [ ]:
# Поиск Kaggle MAT и строго manual_mask при любой вложенности Input.
from collections import Counter

def common_parent(paths):
    return Counter(p.parent for p in paths).most_common(1)[0][0] if paths else None

def belongs_to_irt_dataset(path):
    normalized = str(path).lower().replace('_', '-')
    return 'irt-pvc-depth' in normalized

INPUT_ROOT = Path('/kaggle/input')
all_mats = sorted(INPUT_ROOT.rglob('*.mat'))
kaggle_mats = [p for p in all_mats if belongs_to_irt_dataset(p)] or all_mats
manual_masks = sorted(
    p for p in INPUT_ROOT.rglob('*.png')
    if 'manual_mask' in str(p).lower() and belongs_to_irt_dataset(p)
)
assert kaggle_mats, 'Не найдены MAT: добавьте ziangwei/irt-pvc-depth через Add Input'
assert manual_masks, 'Не найден каталог manual_mask в Kaggle Input'

mat_stems = {p.stem for p in kaggle_mats}
mask_stems = {p.stem for p in manual_masks}
expected_video_ids = {
    *(f'R_{index:03d}' for index in range(2, 21)),
    *(f'Z_{index:03d}' for index in range(2, 21)),
}
assert mat_stems == expected_video_ids, (
    f'Ожидались ровно 38 видео из repo index; '
    f'нет={sorted(expected_video_ids - mat_stems)}, '
    f'лишние={sorted(mat_stems - expected_video_ids)}'
)
missing_masks = sorted(mat_stems - mask_stems)
assert not missing_masks, f'Нет manual_mask для видео: {missing_masks}'

KAGGLE_DATA_DIR = common_parent(kaggle_mats)
KAGGLE_MASK_DIR = common_parent(manual_masks)
print('Kaggle:', len(kaggle_mats), 'videos |', len(manual_masks), 'manual masks')
print('MAT:', KAGGLE_DATA_DIR)
print('MASK:', KAGGLE_MASK_DIR)
print('Dataset audit OK: R_002..R_020 + Z_002..Z_020')


In [ ]:
# Поиск Yandex/TPU MAT и создание индивидуальных файлов из общей base_mask.
# Добавьте свой загруженный Yandex dataset через Kaggle Add Input.
import base64
import numpy as np
from PIL import Image

import re
import requests

YANDEX_PUBLIC_URL = 'https://disk.yandex.ru/d/POr5765WUdKLbg'
# /kaggle/temp не включается в сохраняемый Notebook Output.
YANDEX_DOWNLOAD_DIR = Path('/kaggle/temp/yandex-tpu')
YANDEX_DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
PUBLIC_API = 'https://cloud-api.yandex.net/v1/disk/public/resources'

def public_files(path=None):
    params = {'public_key': YANDEX_PUBLIC_URL, 'limit': 1000}
    if path is not None:
        params['path'] = path
    response = requests.get(PUBLIC_API, params=params, timeout=60)
    response.raise_for_status()
    resource = response.json()
    if resource.get('type') == 'file':
        yield resource
        return
    for item in resource.get('_embedded', {}).get('items', []):
        if item.get('type') == 'dir':
            yield from public_files(item['path'])
        else:
            yield item

remote_mats = sorted(
    (
        item for item in public_files()
        if item.get('name', '').lower().endswith('.mat')
    ),
    key=lambda item: item['name'].lower(),
)
assert remote_mats, 'По публичной ссылке Yandex не найдены MAT-файлы'

for number, item in enumerate(remote_mats, start=1):
    destination = YANDEX_DOWNLOAD_DIR / item['name']
    expected_size = int(item.get('size') or 0)
    if (
        destination.exists()
        and destination.stat().st_size > 0
        and (not expected_size or destination.stat().st_size == expected_size)
    ):
        print(f'Yandex cache {number}/{len(remote_mats)}: {destination.name}')
        continue

    download_url = item.get('file')
    if not download_url:
        metadata = requests.get(
            PUBLIC_API,
            params={'public_key': YANDEX_PUBLIC_URL, 'path': item['path']},
            timeout=60,
        )
        metadata.raise_for_status()
        download_url = metadata.json()['file']

    partial = destination.with_suffix(destination.suffix + '.part')
    print(f'Yandex download {number}/{len(remote_mats)}: {destination.name}')
    with requests.get(download_url, stream=True, timeout=300) as response:
        response.raise_for_status()
        with partial.open('wb') as output:
            for chunk in response.iter_content(chunk_size=8 * 1024 * 1024):
                if chunk:
                    output.write(chunk)
    if expected_size and partial.stat().st_size != expected_size:
        raise IOError(
            f'Файл {destination.name} скачан не полностью: '
            f'{partial.stat().st_size} из {expected_size} байт'
        )
    partial.replace(destination)

yandex_mats = sorted(
    path for path in YANDEX_DOWNLOAD_DIR.glob('*.mat')
    if 'sample' in path.stem.lower()
)
assert yandex_mats, 'Yandex MAT скачались, но Sample-видео не распознаны'
sample_numbers = sorted({
    int(match.group(1))
    for path in yandex_mats
    if (match := re.search(r'sample\s+(\d+)', path.stem, flags=re.I))
})
missing_samples = sorted(set(range(1, 22)) - set(sample_numbers))
print('Yandex MAT:', len(yandex_mats), '| sample numbers:', sample_numbers)
if missing_samples:
    print('Внимание: в публичной папке отсутствуют номера:', missing_samples)
YANDEX_DATA_DIR = YANDEX_DOWNLOAD_DIR

MASK_URL = (
    'https://raw.githubusercontent.com/tomatoCoderq/'
    'thermal-control-ya-project/main/datasets/dataset_tpu/base_mask.png'
)
MASK_SHA256 = 'e3a396a32027aa61712dfcafcf8b9c6f2510ba5997b8d9cd056098a56672d8a9'
BASE_MASK_PATH = Path('/kaggle/working/thermal/base_mask.png')
BASE_MASK_PATH.parent.mkdir(parents=True, exist_ok=True)

def file_sha256(path):
    import hashlib
    digest = hashlib.sha256()
    with path.open('rb') as source:
        for chunk in iter(lambda: source.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

if not BASE_MASK_PATH.exists() or file_sha256(BASE_MASK_PATH) != MASK_SHA256:
    print('Скачиваем единую base_mask.png с GitHub')
    response = requests.get(MASK_URL, timeout=60)
    response.raise_for_status()
    partial_mask = BASE_MASK_PATH.with_suffix('.png.part')
    partial_mask.write_bytes(response.content)
    if file_sha256(partial_mask) != MASK_SHA256:
        raise IOError('GitHub base_mask.png не прошла проверку SHA-256')
    partial_mask.replace(BASE_MASK_PATH)
else:
    print('Mask cache:', BASE_MASK_PATH)

base_image = np.asarray(Image.open(BASE_MASK_PATH))
base_binary = (
    base_image[..., 1] > 0 if base_image.ndim == 3 else base_image > 0
).astype(np.uint8)
assert base_binary.shape == (240, 320), base_binary.shape
assert 0 < int(base_binary.sum()) < base_binary.size

YANDEX_MASK_DIR = Path('/kaggle/working/thermal/yandex_base_masks')
YANDEX_MASK_DIR.mkdir(parents=True, exist_ok=True)
for mat_path in yandex_mats:
    video_id = mat_path.stem.replace(' ', '_')
    Image.fromarray(base_binary * 255).save(YANDEX_MASK_DIR / f'{video_id}.png')

print('Yandex/TPU:', len(yandex_mats), 'videos')
print('MAT:', YANDEX_DATA_DIR)
print('Shared experimental mask:', BASE_MASK_PATH)
print('Per-video mask aliases:', YANDEX_MASK_DIR)


In [ ]:
# Общие параметры: одинаковы во всех трёх экспериментах.
KAGGLE_ROI = {'x': 80, 'y': 25, 'w': 178, 'h': 190}
YANDEX_ROI = {'x': 80, 'y': 43, 'w': 178, 'h': 120}
PER_VIDEO_ROIS = {
    # 'R_002': {'x': 82, 'y': 27, 'w': 175, 'h': 185},
}

MODEL_B2 = 'nvidia/segformer-b2-finetuned-ade-512-512'
EXPERIMENT_SEED = 67
EPOCHS = 80
BATCH_SIZE = 4
TRAIN_VIEWS_PER_VIDEO = 12
HEAD_WARMUP_EPOCHS = 5
ENCODER_LR = 1e-5
HEAD_LR = 5e-5
WARMUP_HEAD_LR = 1e-4
PATCH_REPLACEMENT_P = 0.5

# При повторном Run All готовые результаты не обучаются заново.
FORCE_RETRAIN = False

EXPERIMENTS = [
    {'name': 'b2_tsr_manual',  'feature_mode': 'tsr', 'model_name': MODEL_B2, 'enabled': True},
    {'name': 'b2_pca3_manual', 'feature_mode': 'pca', 'model_name': MODEL_B2, 'enabled': True},
    {'name': 'b2_ppt123_manual', 'feature_mode': 'ppt', 'model_name': MODEL_B2, 'enabled': True},
]
print('Будут запущены:', [e['name'] for e in EXPERIMENTS if e['enabled']])


In [ ]:
# Генерация трёх run-конфигов с общими split/аугментациями/оптимизацией.
from copy import deepcopy
import json, yaml

RUN_ROOT = Path('/kaggle/working/thermal/segformer_feature_comparison_manual')
CONFIG_ROOT = RUN_ROOT / 'configs'
CONFIG_ROOT.mkdir(parents=True, exist_ok=True)
(RUN_ROOT / 'dataset_audit.json').write_text(json.dumps({
    'train_validation_source': {
        'source': 'ziangwei/irt-pvc-depth',
        'data_dir': str(KAGGLE_DATA_DIR),
        'mask_dir': str(KAGGLE_MASK_DIR),
        'mask_type': 'manual_mask',
        'video_ids': sorted(mat_stems),
        'n_videos': len(mat_stems),
    },
    'external_test_source': {
        'source': 'yandex/tpu',
        'data_dir': str(YANDEX_DATA_DIR),
        'mask': str(BASE_MASK_PATH),
        'video_files': [path.name for path in yandex_mats],
        'n_videos': len(yandex_mats),
    },
}, ensure_ascii=False, indent=2), encoding='utf-8')

def build_dataset_config(feature_mode):
    template_path = (
        Path('configs/augmentation_tsr.yaml')
        if feature_mode == 'tsr'
        else Path('configs/augmentation_ppt.yaml')
    )
    dataset_cfg = deepcopy(yaml.safe_load(template_path.read_text())['dataset'])
    if feature_mode == 'pca':
        dataset_cfg['features'].update({
            'extractors': ['pca'], 'pca_components': 3,
            'frame_step': 4, 'max_frames': None, 'thermal_diff': True,
        })
    elif feature_mode == 'ppt':
        dataset_cfg['features'].update({
            'extractors': ['ppt'], 'ppt_bins': [1, 2, 3],
            'ppt_frames': 512, 'ppt_auto_cooling': True,
        })

    dataset_cfg['sources'] = [{
        'root': str(KAGGLE_DATA_DIR), 'masks': str(KAGGLE_MASK_DIR),
        'pattern': '*.mat', 'object_roi': KAGGLE_ROI,
    }]
    dataset_cfg['seed'] = EXPERIMENT_SEED
    dataset_cfg['samples_per_video'] = TRAIN_VIEWS_PER_VIDEO
    dataset_cfg['files_meta'] = {
        key: {'object_roi': value} for key, value in PER_VIDEO_ROIS.items()
    }
    dataset_cfg['cache_dir'] = str(RUN_ROOT / 'video_cache')
    dataset_cfg['features']['cache_dir'] = str(RUN_ROOT / f'{feature_mode}_features')
    dataset_cfg['loader']['batch_size'] = BATCH_SIZE
    dataset_cfg['loader']['num_workers'] = 4
    dataset_cfg['loader']['pin_memory'] = True
    for spec in dataset_cfg['augs']['spatial']:
        if spec['name'] == 'PatchReplacement':
            spec['params']['p'] = PATCH_REPLACEMENT_P
    return dataset_cfg

def build_training_config(experiment):
    return {
        'model_name': experiment['model_name'], 'pretrained': True,
        'epochs': EPOCHS, 'head_warmup_epochs': HEAD_WARMUP_EPOCHS,
        'warmup_head_learning_rate': WARMUP_HEAD_LR,
        'encoder_learning_rate': ENCODER_LR, 'head_learning_rate': HEAD_LR,
        'weight_decay': 0.01, 'dice_weight': 0.5,
        'lr_factor': 0.5, 'lr_patience': 4, 'min_learning_rate': 1e-7,
        'early_stopping_patience': 12, 'early_stopping_min_delta': 1e-4,
        'flip_tta': True, 'threshold_min': 0.25,
        'threshold_max': 0.75, 'threshold_steps': 21,
        'val_fraction': 0.15, 'test_fraction': 0.15,
        'split_seed': EXPERIMENT_SEED,
        'amp': True, 'precompute_features': True,
        'output_dir': str(RUN_ROOT / experiment['name']),
    }

def build_external_dataset_config(run):
    external = deepcopy(run['dataset'])
    external['train'] = False
    external['samples_per_video'] = 1
    external['files_meta'] = {}
    external['sources'] = [{
        'root': str(YANDEX_DATA_DIR), 'masks': str(YANDEX_MASK_DIR),
        'pattern': '*.mat', 'object_roi': YANDEX_ROI,
    }]
    mode = run['feature_mode']
    external['cache_dir'] = str(RUN_ROOT / 'yandex_video_cache')
    external['features']['cache_dir'] = str(RUN_ROOT / f'yandex_{mode}_features')
    external['loader']['shuffle'] = False
    return external

RUNS = []
for experiment in EXPERIMENTS:
    if not experiment['enabled']:
        continue
    dataset_cfg = build_dataset_config(experiment['feature_mode'])
    training_cfg = build_training_config(experiment)
    config_path = CONFIG_ROOT / f"{experiment['name']}.yaml"
    config_path.write_text(
        yaml.safe_dump({'dataset': dataset_cfg, 'training': training_cfg}, sort_keys=False),
        encoding='utf-8',
    )
    RUNS.append({**experiment, 'dataset': dataset_cfg,
                 'training': training_cfg, 'config_path': config_path})

print('Конфиги:')
for run in RUNS:
    print(' ', run['name'], '->', run['config_path'])


In [ ]:
# Sanity check Kaggle train-domain и Yandex external test-domain.
from irt_data.config import DatasetConfig
from irt_data.dataset import IRTDataset
import matplotlib.pyplot as plt

fig, axes = plt.subplots(len(RUNS), 4, figsize=(14, 4 * len(RUNS)))
if len(RUNS) == 1:
    axes = axes[None, :]

for row, run in enumerate(RUNS):
    check = deepcopy(run['dataset'])
    check['train'] = False
    check['samples_per_video'] = 1
    check['sources'][0]['pattern'] = kaggle_mats[0].name
    sample = IRTDataset(DatasetConfig.from_dict(check))[0]
    assert tuple(sample['image'].shape) == (3, 256, 256)
    assert tuple(sample['mask'].shape) == (256, 256)
    assert set(sample['mask'].unique().tolist()) <= {0, 1}
    print(run['name'], tuple(sample['image'].shape), sample['video_id'])
    for channel in range(3):
        axes[row, channel].imshow(sample['image'][channel], cmap='inferno')
        axes[row, channel].set_title(f"{run['feature_mode']} channel {channel}")
    axes[row, 3].imshow(sample['mask'], cmap='gray')
    axes[row, 3].set_title('manual mask')
    for axis in axes[row]: axis.axis('off')
plt.tight_layout()
plt.show()

# Один Yandex sample: feature maps и base_mask должны пройти один ROI/resize.
external_check = build_external_dataset_config(RUNS[0])
external_check['sources'][0]['pattern'] = yandex_mats[0].name
external_sample = IRTDataset(DatasetConfig.from_dict(external_check))[0]
assert tuple(external_sample['image'].shape) == (3, 256, 256)
assert tuple(external_sample['mask'].shape) == (256, 256)
assert set(external_sample['mask'].unique().tolist()) <= {0, 1}
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(external_sample['image'][0], cmap='inferno')
axes[0].set_title(f"Yandex feature: {external_sample['video_id']}")
axes[1].imshow(external_sample['mask'], cmap='gray')
axes[1].set_title('Yandex shared base_mask after ROI')
for axis in axes: axis.axis('off')
plt.tight_layout(); plt.show()


In [ ]:
# Общая функция запуска. Сама эта ячейка модели НЕ обучает.
import json, shutil, subprocess, sys
from torch.utils.data import DataLoader

from irt_data.loaders import stack_collate, worker_init_fn
from segformer.model import build_segformer
from train_segformer import evaluate, precompute_features

trainer_path = Path('train_segformer.py')
trainer_text = trainer_path.read_text(encoding='utf-8')
device_line = '    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")'
if 'TRAIN_DEVICE=' not in trainer_text:
    replacement = device_line + '\n    if device.type != "cuda":\n        raise RuntimeError("CUDA is required, but child process cannot see a GPU")\n    print(f"TRAIN_DEVICE={device}; GPU={torch.cuda.get_device_name(0)}", flush=True)'
    assert device_line in trainer_text
    trainer_path.write_text(trainer_text.replace(device_line, replacement), encoding='utf-8')

training_env = os.environ.copy()
training_env['CUDA_VISIBLE_DEVICES'] = '0'
training_env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
gpu_probe = 'import torch; assert torch.cuda.is_available(); print("CHILD_GPU=" + torch.cuda.get_device_name(0))'
subprocess.check_call([sys.executable, '-c', gpu_probe], env=training_env)

RUN_BY_NAME = {run['name']: run for run in RUNS}

def evaluate_external_yandex(run):
    output_dir = Path(run['training']['output_dir'])
    metrics_path = output_dir / 'yandex_test_metrics.json'
    if metrics_path.exists() and not FORCE_RETRAIN:
        print(f"SKIP Yandex test {run['name']}: результаты уже существуют")
        return json.loads(metrics_path.read_text())

    external_cfg_raw = build_external_dataset_config(run)
    external_cfg = DatasetConfig.from_dict(external_cfg_raw)
    external_dataset = IRTDataset(external_cfg)
    print(f"Yandex feature cache: {run['name']} ({len(external_dataset.video_ids)} videos)")
    precompute_features(external_dataset)
    external_loader = DataLoader(
        external_dataset,
        batch_size=external_cfg.loader.batch_size,
        shuffle=False,
        num_workers=external_cfg.loader.num_workers,
        pin_memory=external_cfg.loader.pin_memory,
        collate_fn=stack_collate,
        worker_init_fn=worker_init_fn if external_cfg.loader.num_workers > 0 else None,
    )

    device = torch.device('cuda')
    model = build_segformer(run['model_name'], pretrained=True).to(device)
    checkpoint = torch.load(output_dir / 'best.pt', map_location=device, weights_only=True)
    model.load_state_dict(checkpoint['model'])
    inference = json.loads((output_dir / 'inference_config.json').read_text())
    metrics = evaluate(
        model, external_loader, device,
        threshold=float(inference['threshold']),
        flip_tta=bool(inference['flip_tta']),
        dice_weight=float(run['training']['dice_weight']),
    )
    metrics.update({
        'domain': 'yandex_external_test',
        'n_videos': len(external_dataset.video_ids),
        'threshold_from_kaggle_validation': float(inference['threshold']),
        'flip_tta': bool(inference['flip_tta']),
        'video_ids': list(external_dataset.video_ids),
    })
    metrics_path.write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    del model, external_loader, external_dataset
    torch.cuda.empty_cache()
    print(f"YANDEX TEST {run['name']}:", json.dumps(metrics, indent=2))
    return metrics

def run_experiment(name):
    run = RUN_BY_NAME[name]
    output_dir = Path(run['training']['output_dir'])
    metrics_path = output_dir / 'test_metrics.json'
    if metrics_path.exists() and not FORCE_RETRAIN:
        print(f"SKIP Kaggle training {name}: результаты уже существуют")
        kaggle_metrics = json.loads(metrics_path.read_text())
    else:
        output_dir.mkdir(parents=True, exist_ok=True)
        log_path = output_dir / 'train.log'
        print(f"\n{'=' * 80}\nSTART KAGGLE TRAIN {name}\n{'=' * 80}")
        command = [sys.executable, 'train_segformer.py', '--config', str(run['config_path'])]
        with log_path.open('w', encoding='utf-8') as log_file:
            process = subprocess.Popen(
                command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                text=True, bufsize=1, env=training_env,
            )
            for line in process.stdout:
                print(line, end='')
                log_file.write(line)
            return_code = process.wait()
        if return_code != 0:
            raise RuntimeError(f"Ошибка {name}, см. {log_path}")
        shutil.copy2(run['config_path'], output_dir / 'run_config.yaml')
        kaggle_metrics = json.loads(metrics_path.read_text())
        print(f"DONE KAGGLE TRAIN {name}:", json.dumps(kaggle_metrics, indent=2))

    yandex_metrics = evaluate_external_yandex(run)
    return {'kaggle_internal_test': kaggle_metrics, 'yandex_external_test': yandex_metrics}


## Эксперимент 1 — SegFormer-B2 + TSR + manual masks

Долгая ячейка: сначала строится TSR feature cache на CPU, затем обучается B2 на GPU.


In [ ]:
tsr_metrics = run_experiment('b2_tsr_manual')
tsr_metrics


## Эксперимент 2 — SegFormer-B2 + PCA3 + manual masks

Независимый запуск на том же split. PCA-кэш хранится отдельно от TSR.


In [ ]:
pca_metrics = run_experiment('b2_pca3_manual')
pca_metrics


## Эксперимент 3 — SegFormer-B2 + PPT/Fourier + manual masks

Независимый запуск с phase maps первых трёх ненулевых Fourier bins.


In [ ]:
ppt_metrics = run_experiment('b2_ppt123_manual')
ppt_metrics


In [ ]:
# Общая таблица: Kaggle internal reference и главный Yandex external test.
import json
import pandas as pd

rows = []
for run in RUNS:
    output = Path(run['training']['output_dir'])
    if not (output / 'test_metrics.json').exists():
        print('Ещё не обучен:', run['name'])
        continue
    tuned = json.loads((output / 'test_metrics.json').read_text())
    raw = json.loads((output / 'test_metrics_default.json').read_text())
    yandex_path = output / 'yandex_test_metrics.json'
    if not yandex_path.exists():
        print('Ещё не проверен на Yandex:', run['name'])
        continue
    yandex = json.loads(yandex_path.read_text())
    history = json.loads((output / 'history.json').read_text())
    best_val = max(history, key=lambda item: item['val_dice'])
    rows.append({
        'experiment': run['name'],
        'features': run['feature_mode'],
        'best_epoch': best_val['epoch'],
        'best_val_dice': best_val['val_dice'],
        'kaggle_raw_dice': raw['dice'], 'kaggle_raw_iou': raw['iou'],
        'kaggle_tuned_dice': tuned['dice'], 'kaggle_tuned_iou': tuned['iou'],
        'yandex_dice': yandex['dice'], 'yandex_iou': yandex['iou'],
        'yandex_precision': yandex['precision'], 'yandex_recall': yandex['recall'],
        'threshold': tuned['threshold'], 'flip_tta': tuned['flip_tta'],
        'yandex_videos': yandex['n_videos'],
    })

assert rows, 'Сначала выполните хотя бы одну из трёх ячеек обучения'
comparison = pd.DataFrame(rows).sort_values('yandex_dice', ascending=False)
display(comparison.style.format({
    column: '{:.4f}' for column in comparison.columns
    if column not in {'experiment', 'features', 'best_epoch', 'flip_tta', 'yandex_videos'}
}))
comparison.to_csv(RUN_ROOT / 'comparison.csv', index=False)
comparison.to_json(RUN_ROOT / 'comparison.json', orient='records', indent=2)


In [ ]:
# Кривые validation Dice/IoU для трёх методов на одном графике.
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for run in RUNS:
    output = Path(run['training']['output_dir'])
    if not (output / 'history.json').exists():
        continue
    history = json.loads((output / 'history.json').read_text())
    epochs = [item['epoch'] for item in history]
    axes[0].plot(epochs, [item['val_dice'] for item in history], label=run['feature_mode'])
    axes[1].plot(epochs, [item['val_iou'] for item in history], label=run['feature_mode'])
axes[0].set_title('Validation Dice')
axes[1].set_title('Validation IoU')
for axis in axes:
    axis.set_xlabel('epoch'); axis.grid(); axis.legend()
plt.tight_layout()
plot_path = RUN_ROOT / 'validation_comparison.png'
plt.savefig(plot_path, dpi=160, bbox_inches='tight')
plt.show()
print('График:', plot_path)


In [ ]:
# Один архив со всеми checkpoints, split, логами, конфигами и таблицей.
import shutil
archive_path = shutil.make_archive(
    '/kaggle/working/segformer_b2_kaggle_train_yandex_test_results',
    'zip', root_dir=RUN_ROOT,
)
print('Скачайте:', archive_path)


## Как читать результат

- Главные столбцы — `yandex_dice`, `yandex_iou`, `yandex_precision` и
  `yandex_recall`: это перенос Kaggle → Yandex без дообучения на Yandex.
- Threshold выбирается только на Kaggle validation и затем фиксируется для
  Yandex. Yandex labels не участвуют в выборе checkpoint или threshold.
- `kaggle_*` — внутренняя reference-оценка, чтобы отделить качество модели от
  domain shift между двумя экспериментальными установками.
- Общая `base_mask` корректна при условии, что геометрия дефектов действительно
  одинакова во всех Yandex-опытах.
- Увеличение числа аугментированных views не создаёт новые независимые видео.
